# Ghost Data Still Exists — Google Colab (free tier)

Run cells in order. Uses public open-weight models on the allocated Colab machine, with **no paid API calls**. Model downloads need Internet but inference is local. Free GPU availability and runtime duration vary; do not upgrade or subscribe for this experiment.

Start with CPU preparation, then choose **Runtime → Change runtime type → GPU** for inference. Changing runtime can reset files; rerun setup if needed. Optionally save caches to Google Drive below. Model accuracy and memory use on Colab have not yet been measured.


In [ ]:
from pathlib import Path
import subprocess, sys, json, os
PROJECT = Path('/content/project')
REPO = 'https://github.com/abdel-saw/hackerrank-orchestrate-october26.git'
BRANCH = 'work'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, str(PROJECT)], check=True)
else:
    assert (PROJECT / 'code/main.py').is_file(), 'Existing /content/project is not this project; preserve your files and choose another PROJECT path.'
    print('Reusing existing checkout; no reset or pull performed.')
DATASET = PROJECT / 'dataset'
assert (DATASET / 'deletion_requests.csv').is_file()
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=PROJECT, check=True)
print('Project:', PROJECT)


## Install prerequisites

Keep Colab's preinstalled PyTorch/CUDA build. Package installation and public weight downloads are setup operations, not paid inference services. If pip reports a runtime restart is required, restart and rerun setup cells.


In [ ]:
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'tesseract-ocr'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'code/requirements-colab.txt')], check=True)
run('-m', 'unittest', 'discover', '-s', 'code/tests', '-v')


## Choose cache storage

Set `USE_DRIVE = True` to mount your Drive through Colab's normal authorization flow and retain extraction caches, downloaded weights, responses and reports between runtimes. Only the named project folder is used. Without Drive, download outputs before the runtime ends. Never place credentials in notebook cells.


In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STORAGE = Path('/content/drive/MyDrive/ghost-data-audit')
else:
    STORAGE = Path('/content/ghost-data-audit')
STORAGE.mkdir(parents=True, exist_ok=True)
ARTIFACTS = STORAGE / 'artifacts'
print('Cache/output storage:', STORAGE)


## Prepare evidence on CPU

No model is used here. A successful extraction does not establish OCR accuracy. Inspect `coverage.json` and sample packets; do not equate missing evidence with deletion completion.


In [ ]:
run('code/main.py', '--dataset', DATASET, '--artifacts', ARTIFACTS / 'sample')
coverage = json.loads((ARTIFACTS / 'sample/coverage.json').read_text())
gaps = [row for row in coverage if row['status'] != 'extracted']
print('Evidence files:', len(coverage), 'Extraction gaps:', len(gaps))
for gap in gaps:
    print(gap)
assert not gaps, 'Resolve extraction gaps before inference'


## Check GPU and download a small local model

Initial candidate: **Qwen/Qwen2.5-3B-Instruct**, in 4-bit precision. This is a starting experiment, not a measured best model. The download is several GB. The selected Hugging Face repository is public; no API key is required. Its exact revision is saved on first download and reused for repeatability. Review the model license before use.

If no GPU is available on the free tier, CPU evidence preparation still works. Resume inference when a free GPU is available. Do not remove the guard and accidentally start a long CPU inference run.


In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU available. Select a GPU runtime, or resume later when a free GPU is available.'
print('GPU:', torch.cuda.get_device_name(0))
print('Free/total GPU bytes:', torch.cuda.mem_get_info())
from huggingface_hub import HfApi, snapshot_download
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
MANIFEST = STORAGE / 'model_revision.json'
if MANIFEST.exists():
    model_info = json.loads(MANIFEST.read_text())
    assert model_info['model_id'] == MODEL_ID, 'Choose a new storage directory for a different model'
else:
    model_info = {'model_id': MODEL_ID, 'revision': HfApi().model_info(MODEL_ID).sha}
    MANIFEST.write_text(json.dumps(model_info, indent=2))
MODEL = Path(snapshot_download(
    repo_id=MODEL_ID, revision=model_info['revision'],
    cache_dir=str(STORAGE / 'hf-cache'),
    allow_patterns=['*.json', '*.safetensors', '*.model', 'tokenizer*', 'vocab*', 'merges.txt', '*.tiktoken'],
))
print('Model:', model_info)
print('Local model directory:', MODEL)


## Run the 32 public samples

Inference uses only local files. Each case's response is cached immediately. A disconnect can interrupt a run, but previously cached responses can be reused after restart when storage persists. Context overflow or unsupported model answers block final output rather than create invented predictions. This first version processes cases one at a time.

If this cell fails after inference, inspect the saved diagnostics in the next cell. GPU out-of-memory errors require reducing context or choosing a smaller model; never silently truncate relevant evidence.


In [ ]:
SAMPLE_OUTPUT = STORAGE / 'sample_predictions.csv'
run('code/main.py', '--dataset', DATASET, '--artifacts', ARTIFACTS / 'sample',
    '--model-path', MODEL, '--quantize', '--output', SAMPLE_OUTPUT)


In [ ]:
failure_path = ARTIFACTS / 'sample/failures.json'
if failure_path.exists():
    failures = json.loads(failure_path.read_text())
    print('Failed sample cases:', len(failures))
    for failure in failures[:10]:
        print(failure)


## Evaluate samples

This reads sample labels only in the separate evaluator, never in inference. Partial outputs can be evaluated with missing cases penalized, so failures remain visible. Sample accuracy does not establish hidden-set accuracy. Review per-case differences before any full run. Preserve reports from each experiment when comparing models.


In [ ]:
SAMPLE_REPORT = STORAGE / 'reports/sample'
# Always use this run's diagnostic output, avoiding any stale final CSV from an older run.
PREDICTIONS = ARTIFACTS / 'sample/partial_predictions.csv'
assert PREDICTIONS.is_file(), 'Inference has not completed enough to save predictions; inspect its error first'
run('code/evaluation/main.py', '--reference', DATASET / 'sample_output.csv',
    '--predictions', PREDICTIONS, '--report', SAMPLE_REPORT,
    '--usage', ARTIFACTS / 'sample/usage.json')
print((SAMPLE_REPORT / 'evaluation_report.md').read_text()[:6000])


## Optional full run

Enable only after examining sample errors. Free-tier limits may interrupt a long run. There are no visible labels for these 518 cases. Do not evaluate their predictions against the sample labels. The sample usage report is not the final full-run usage report: preserve the full usage ledger and prepare the required Markdown report from that run.


In [ ]:
RUN_FULL = False
if RUN_FULL:
    run('code/main.py', '--dataset', DATASET, '--requests', 'deletion_requests.csv',
        '--artifacts', ARTIFACTS / 'full', '--model-path', MODEL, '--quantize',
        '--output', PROJECT / 'output.csv')
    import shutil
    shutil.copy2(PROJECT / 'output.csv', STORAGE / 'output.csv')


## Download experiment results

Creates an experiment-results archive, **not final submission `code.zip`**. It contains model revision metadata, reports, per-case diagnostics and usage if available. It excludes model weights and large extraction/prompt caches. With Drive enabled these files already persist in STORAGE. Model response caches are retained in STORAGE but excluded from this download to keep it small.

Final submission also needs runnable source, investigated evaluation failures and improvements, at least three additional authored evaluation cases, final full-run usage, `log.txt`, and the separately requested development traces. Never include model weights or the dataset in `code.zip`.


In [ ]:
import zipfile
from google.colab import files
archive = Path('/content/ghost-audit-results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in STORAGE.rglob('*'):
        if not p.is_file():
            continue
        rel = p.relative_to(STORAGE)
        if any(part in {'hf-cache', 'extraction_cache', 'model_cache'} for part in rel.parts):
            continue
        if p.name == 'packets.jsonl':
            continue
        z.write(p, str(rel))
files.download(str(archive))
